Importing the Dependencies

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn import svm
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

Data Collection and Analysis

Diabetes Dataset

In [2]:
diabetes_dataset = pd.read_csv('../Dataset/diabetes.csv')

In [3]:
# Convert invalid zero values to NaN (domain knowledge: these columns cannot have zero values)
cols_with_invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_with_invalid_zeros:
    diabetes_dataset[col] = diabetes_dataset[col].replace(0, np.nan)

print("Missing values after zero-to-NaN conversion:")
print(diabetes_dataset[cols_with_invalid_zeros].isnull().sum())

Missing values after zero-to-NaN conversion:
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64


In [4]:
diabetes_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   763 non-null    float64
 2   BloodPressure             733 non-null    float64
 3   SkinThickness             541 non-null    float64
 4   Insulin                   394 non-null    float64
 5   BMI                       757 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(6), int64(3)
memory usage: 54.1 KB


In [5]:
X = diabetes_dataset.drop(columns = 'Outcome')
Y = diabetes_dataset['Outcome']

In [6]:
print(Y)

0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: Outcome, Length: 768, dtype: int64


In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.2, stratify=Y, random_state=2)

In [8]:
# Create Pipeline: Imputer -> Scaler -> SVC
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', svm.SVC(kernel='linear'))
])

print("Pipeline created:")
print(pipeline)

Pipeline created:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier', SVC(kernel='linear'))])


In [9]:
# Fit Pipeline on training data only (imputer and scaler learn from X_train only)
pipeline.fit(X_train, Y_train)
print("Pipeline fitted on training data")

Pipeline fitted on training data


In [10]:
# Model Evaluation using Pipeline (handles imputation + scaling + prediction internally)
X_train_prediction = pipeline.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

print(f'Accuracy score of the training data : {training_data_accuracy*100:.2f}%')

Accuracy score of the training data : 77.85%


### Model Evaluation

Evaluating the Pipeline on the test split and through 5-Fold Stratified Cross-Validation.

In [11]:
# Comprehensive Model Evaluation on Test Data
X_test_prediction = pipeline.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)

print(f'Accuracy score of the test data : {test_data_accuracy*100:.2f}%')

# Confusion Matrix
cm = confusion_matrix(Y_test, X_test_prediction)
print('\nConfusion Matrix:')
print(cm)
print(f'  True Negatives: {cm[0,0]}, False Positives: {cm[0,1]}')
print(f'  False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}')

# Classification Report
print('\nClassification Report:')
print(classification_report(Y_test, X_test_prediction, target_names=['Non-Diabetic (0)', 'Diabetic (1)']))

# Individual Metrics
precision = precision_score(Y_test, X_test_prediction)
recall = recall_score(Y_test, X_test_prediction)
f1 = f1_score(Y_test, X_test_prediction)
y_scores = pipeline.decision_function(X_test)
roc_auc = roc_auc_score(Y_test, y_scores)

print(f'Precision: {precision:.4f}')
print(f'Recall (Sensitivity): {recall:.4f}')
print(f'F1-Score: {f1:.4f}')
print(f'ROC-AUC Score: {roc_auc:.4f}')

Accuracy score of the test data : 77.27%

Confusion Matrix:
[[91  9]
 [26 28]]
  True Negatives: 91, False Positives: 9
  False Negatives: 26, True Positives: 28

Classification Report:
                  precision    recall  f1-score   support

Non-Diabetic (0)       0.78      0.91      0.84       100
    Diabetic (1)       0.76      0.52      0.62        54

        accuracy                           0.77       154
       macro avg       0.77      0.71      0.73       154
    weighted avg       0.77      0.77      0.76       154

Precision: 0.7568
Recall (Sensitivity): 0.5185
F1-Score: 0.6154
ROC-AUC Score: 0.8200


### Stratified K-Fold Cross-Validation

Validating the Pipeline across 5 folds to evaluate generalization performance.

In [12]:
# 5-Fold Stratified Cross-Validation on the full dataset using the Pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

cv_acc = cross_val_score(pipeline, X, Y, cv=cv, scoring='accuracy')
cv_precision = cross_val_score(pipeline, X, Y, cv=cv, scoring='precision')
cv_recall = cross_val_score(pipeline, X, Y, cv=cv, scoring='recall')
cv_f1 = cross_val_score(pipeline, X, Y, cv=cv, scoring='f1')
cv_roc = cross_val_score(pipeline, X, Y, cv=cv, scoring='roc_auc')

print('=== 5-Fold Stratified Cross-Validation Results ===')
print(f'Accuracy : {cv_acc.mean()*100:.2f}% (+/- {cv_acc.std()*100:.2f}%)')
print(f'Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})')
print(f'Recall   : {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})')
print(f'F1-Score : {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')
print(f'ROC-AUC  : {cv_roc.mean():.4f} (+/- {cv_roc.std():.4f})')

=== 5-Fold Stratified Cross-Validation Results ===
Accuracy : 76.43% (+/- 2.59%)
Precision: 0.7050 (+/- 0.0534)
Recall   : 0.5630 (+/- 0.0702)
F1-Score : 0.6233 (+/- 0.0499)
ROC-AUC  : 0.8291 (+/- 0.0379)


In [13]:
# Making a Predictive System using the Pipeline
input_data = (5,166,72,19,175,25.8,0.587,51)

columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
           'Insulin','BMI','DiabetesPedigreeFunction','Age']

input_df = pd.DataFrame([input_data], columns=columns)

prediction = pipeline.predict(input_df)

print(prediction)

if prediction[0] == 0:
    print('The person is not diabetic')
else:
    print('The person is diabetic')

[1]
The person is diabetic


In [14]:
# Saving the complete Pipeline (imputer + scaler + classifier)
filename = '../saved_models/diabetes_pipeline.joblib'
joblib.dump(pipeline, filename)
print(f'Pipeline saved to {filename}')

Pipeline saved to ../saved_models/diabetes_pipeline.joblib


In [15]:
for column in X.columns:
  print(column)

Pregnancies
Glucose
BloodPressure
SkinThickness
Insulin
BMI
DiabetesPedigreeFunction
Age
